## Import Packages

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Reshape, Conv2D, UpSampling2D
from tensorflow.keras.initializers import he_normal
from tensorflow.keras.optimizers import Adam                                                  
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score                                                         
from pathlib import Path

2025-05-20 12:59:31.848650: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices("GPU"))

[]


## Import Datasets

In [3]:
base_dir = Path('/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Senegal/64x64/full/timesteps')

test_data_X = pd.read_csv(base_dir / 'test-senegal-input-t0-for-lt1.csv')
test_data_y_t1 = np.load(base_dir / '32x32' / 'test-senegal-output-at-lt1-resized.npy')
test_data_y_t0 = np.load(base_dir / '32x32' / 'test-senegal-output-t0-for-lt1-resized.npy')

In [4]:
train_data_X = pd.read_csv(base_dir / 'train-senegal-input-t0-for-lt1.csv')
train_data_y_t1 = np.load(base_dir / '32x32' / 'train-senegal-output-at-lt1-resized.npy')
train_data_y_t0 = np.load(base_dir / '32x32' / 'train-senegal-output-t0-for-lt1-resized.npy')

KeyboardInterrupt: 

### Preprocessing

Log transforming

In [ ]:
train_data_X, val_data_X, train_y_lt1, val_y_lt1 = train_test_split(train_data_X, train_data_y_t1, test_size=0.3, random_state=12)

In [ ]:
# Define all time suffixes
suffixes = ['', '_30', '_60', '_90', '_120']

# Define variable prefixes
prefixes = ['size', 'wp', 'd']
indices = ['1', '2', '3']

# Construct full column names
cols_to_log = [f'{prefix}{i}{suf}' for suf in suffixes for prefix in prefixes for i in indices]

# Apply log1p transform to each dataframe
for col in cols_to_log:
    train_data_X[col] = np.log1p(train_data_X[col].astype(float))
    test_data_X[col] = np.log1p(test_data_X[col].astype(float))
    val_data_X[col] = np.log1p(val_data_X[col].astype(float))

### Scaling

In [ ]:
# Exclude all columns that start with 'mask'
mask_cols = [col for col in train_data_X.columns if col.startswith('mask')]

# Final list of features to scale
cols_to_scale = [col for col in train_data_X.columns if col not in mask_cols]

# Initialise scaler
scaler = StandardScaler()

# Apply scaling
X_train_scaled = train_data_X.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(train_data_X[cols_to_scale])

X_val_scaled = val_data_X.copy()
X_val_scaled[cols_to_scale] = scaler.transform(val_data_X[cols_to_scale])

X_test_scaled = test_data_X.copy()
X_test_scaled[cols_to_scale] = scaler.transform(test_data_X[cols_to_scale])


dump(scaler, "/content/drive/MyDrive/Zambia/t0-only/Zambia-scaler-lt1-t0-only.bin", compress=True)

In [ ]:
assert X_train_scaled.shape[1] == 5 * 23, "Expected 115 features for 5 time steps of 23 features each."

In [ ]:
# Reshape for LSTM: (samples, time_steps=5, features=23)
x_train = X_train_scaled.values.reshape(len(X_train_scaled), 5, 23)
x_val = X_val_scaled.values.reshape(len(X_val_scaled), 5, 23)
x_test = X_test_scaled.values.reshape(len(X_test_scaled), 5, 23)

# Targets
y_train = train_y_lt1
y_val = val_y_lt1
y_test = test_data_y_t1

In [ ]:
input_shape = (5, 23)  # (time_steps, features)
initial_size = 4       # Adjusted to ensure 32x32 output after 3x upsampling

model = tf.keras.models.Sequential([
    Input(shape=(5, 23)),

    # --- Temporal encoding ---
    LSTM(128, return_sequences=True, activation='tanh', recurrent_activation='sigmoid'),
    Dropout(0.2),
    LSTM(64, activation='tanh', recurrent_activation='sigmoid'),
    Dropout(0.2),

    # --- Map projection ---
    Dense(initial_size * initial_size * 16, activation='relu'),
    Reshape((initial_size, initial_size, 16)),

    # --- Upsampling + convolutional decoding ---
    UpSampling2D(size=(2, 2)),  # 4x4 → 8x8
    Conv2D(32, (3, 3), activation='relu', padding='same', kernel_initializer=he_normal()),

    UpSampling2D(size=(2, 2)),  # 8x8 → 16x16
    Conv2D(32, (3, 3), activation='relu', padding='same', kernel_initializer=he_normal()),

    UpSampling2D(size=(2, 2)),  # 16x16 → 32x32
    Conv2D(16, (3, 3), activation='relu', padding='same', kernel_initializer=he_normal()),

    Conv2D(1, (1, 1), activation='sigmoid', padding='same', kernel_initializer=he_normal())  # 32x32x1
])

model.summary()


In [ ]:
cl_thresholds = 0.15

METRICS = [
      tf.keras.metrics.MeanSquaredError(name='bs'),
      tf.keras.metrics.TruePositives(name='tp'),
      tf.keras.metrics.TrueNegatives(name='tn'),
      tf.keras.metrics.Recall(name='recall', thresholds = cl_thresholds),
      tf.keras.metrics.Precision(name='precision', thresholds = cl_thresholds),
      tf.keras.metrics.AUC(name='auc'),
      tf.keras.metrics.AUC(name='prc', curve='PR'),
      tf.keras.metrics.F1Score(name = 'f1_score',  threshold = cl_thresholds),
]

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    verbose=1,
    patience=10,
    mode='max',
    restore_best_weights=True)

In [ ]:
optimizer = Adam(clipvalue=1.0, learning_rate=1e-3)
model.compile(optimizer=optimizer,
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=[tf.keras.metrics.AUC(name='auc')])

In [ ]:
model_history = model.fit(x_train,
                          y_train,
                          epochs=500,
                          batch_size = 1024,
                          callbacks = [early_stopping],
                          validation_data=(x_val, y_val)
                        )

In [ ]:
pred_y_train = model.predict(x_train).flatten()
pred_y_test = model.predict(x_test).flatten()

In [ ]:
def reliability_curve(y_true, y_pred, bin_size=0.1, min_predictions_per_bin=50):
    """
    Computes the reliability curve (calibration curve) for a binary classifier.
    
    Parameters:
        y_true (array-like): Ground truth binary labels (0 or 1).
        y_pred (array-like): Predicted probabilities (between 0 and 1).
        bin_size (float): Width of the bins to divide probability space.
        min_predictions_per_bin (int): Minimum number of predictions in a bin to be included.
        
    Returns:
        bin_centers (list): Midpoints of the bins used.
        bin_positive_rates (list): Observed frequency of positive class in each bin.
        bin_counts (list): Number of predictions in each bin.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    bins = np.arange(0, 1 + bin_size, bin_size)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    bin_positive_rates = []
    bin_centers_output = []
    bin_counts = []

    for lower, upper, center in zip(bins[:-1], bins[1:], bin_centers):
        in_bin = (y_pred >= lower) & (y_pred < upper)
        count_in_bin = np.sum(in_bin)
        
        if count_in_bin >= min_predictions_per_bin:
            observed_rate = np.mean(y_true[in_bin])
            bin_positive_rates.append(round(observed_rate, 3))
            bin_centers_output.append(round(center, 3))
            bin_counts.append(count_in_bin)

    return bin_centers_output, bin_positive_rates, bin_counts


def plot_roc(name, y_true, y_pred_proba, **kwargs):
    """
    Plots the ROC curve with hit rate vs false alarm rate (% scale).
    
    Parameters:
        name (str): Label for the curve (e.g., model name).
        y_true (array-like): Ground truth binary labels (0 or 1).
        y_pred_proba (array-like): Predicted probabilities (from model).
        **kwargs: Additional plotting keyword arguments (e.g. linestyle, color).
    """
    fpr, tpr, _ = metrics.roc_curve(y_true, y_pred_proba)
    
    plt.plot(100 * fpr, 100 * tpr, label=name, linewidth=1.5, **kwargs)
    plt.xlabel('False Alarm Rate [%]')
    plt.ylabel('Hit Rate [%]')
    plt.xlim([-1, 100])
    plt.ylim([0, 105])
    plt.grid(True, linestyle=':', linewidth=0.5)
    plt.legend(loc='lower right')
    plt.savefig(f'/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/output/training/multi-time-steps-32x32/{name}.png')

In [ ]:
auc_train = roc_auc_score(y_train.flatten(), pred_y_train)
auc_test = roc_auc_score(y_test.flatten(), pred_y_test)

In [ ]:
plot_roc(f"Training set (AUC={auc_train:.3f})", y_train.flatten(), pred_y_train.flatten(), color="red")
plot_roc(f"Testing set  (AUC={auc_test:.3f})", y_test.flatten(), pred_y_test.flatten(), linestyle='--')
plt.legend(loc='lower right');

### Reliability diagram

In [ ]:
prob_pred, prob_true, no_pred_per_bin = reliability_curve(y_test.flatten(), pred_y_test.flatten())

plt.figure(figsize=(5,4))
plt.plot(prob_pred, prob_true, label='Hello')

plt.xlabel("Predicted probability")
plt.ylabel("Obs. proportion of positive case")

no_pred_per_bin = [i/(1.1*np.max(no_pred_per_bin)) for i in no_pred_per_bin]
plt.plot(np.arange(0,1.2,0.2), np.arange(0,1.2,0.2))
plt.bar(prob_pred, no_pred_per_bin, width=0.1, linestyle="--", fill=False, edgecolor="green")

plt.xlim(0,1)
plt.ylim(0,1)
plt.legend(loc='upper center')
plt.show()